# ARDY 语音动作复现

在 Colab 连接 T4 后按顺序运行。前三条有效口令驱动路径约束。G1 组对比短窗、长窗和基于 4 帧块的重规划。每轮实测的语音识别时延决定回放时间线。


## 官方代码和 Core 权重

运行环境使用 Python 3.12、NumPy 1.26.4 和 PyTorch 2.8。模型和缓存放在 Colab 的 `/content/ardy-repro/`。


In [ ]:
import concurrent.futures
import os
from pathlib import Path
import subprocess
import urllib.request

base = Path('/content/ardy-repro')
base.mkdir(exist_ok=True)
os.environ['HF_HOME'] = str(base/'hf')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['ARDY_SKIP_MOTION_CORRECTION'] = '1'
commit = 'e706e27acc88070918308a10922b8f7174d0825e'

def run(args):
    subprocess.run(args, check=True)

def install():
    source = base/'official'
    if not (source/'pyproject.toml').exists():
        archive=base/'ardy-source.tar.gz'
        urllib.request.urlretrieve(f'https://codeload.github.com/nv-tlabs/ardy/tar.gz/{commit}',archive)
        source.mkdir(exist_ok=True)
        run(['tar','-xzf',str(archive),'-C',str(source),'--strip-components=1'])
    python=base/'python/bin/python'
    if not python.exists():
        run(['uv','venv','--python','3.12',str(base/'python')])
    run(['uv','pip','install','--python',str(python),'torch==2.8.0','--index-url','https://download.pytorch.org/whl/cu126'])
    run(['uv','pip','install','--python',str(python),'-e',str(source),'faster-whisper==1.2.1','matplotlib'])
    run([str(python),'-c','import torch; import ardy; import numpy; print("ARDY imports OK",torch.__version__,numpy.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))'])

def stage_assets():
    from huggingface_hub import snapshot_download
    model_id = 'nvidia/ARDY-Core-RP-20FPS-Horizon40'
    snapshots = ['config.yaml','stats/*/*.npy','denoiser.safetensors','tokenizer.safetensors','LICENSE']
    snapshot_download(model_id, revision='fe1f897459605c98d5418d036bfcfe73bc244fd8', allow_patterns=snapshots, local_dir=base/'weights'/model_id.split('/')[1], token=False, max_workers=4)
    snapshot_download('Systran/faster-whisper-tiny.en', allow_patterns=['*.json','model.bin','vocabulary.*'], local_dir=base/'whisper-tiny-en', token=False)
    print('Pinned NVIDIA checkpoints and Whisper ready',flush=True)

run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'])
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as ex:
    jobs=[ex.submit(install),ex.submit(stage_assets)]
    for job in jobs: job.result()


## Core 语音 Demo 和 4／10 步对照


In [ ]:
from pathlib import Path
root = Path('/content/ardy-repro')
root.mkdir(exist_ok=True)
(root / "run_ardy.py").write_text("\"\"\"Small, repeatable Colab ARDY playback with explicit speech boundaries.\"\"\"\n\nimport argparse\nfrom datetime import datetime, timezone, timedelta\nimport gc\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport statistics\nimport subprocess\nimport time\n\nimport numpy as np\nimport torch\nfrom ardy.constraints import Root2DConstraintSet\nfrom ardy.model.load_model import load_model, load_text_encoder\nfrom faster_whisper import WhisperModel\n\n\nMODEL = 'ARDY-Core-RP-20FPS-Horizon40'\nCHECKPOINT_REV = 'fe1f897459605c98d5418d036bfcfe73bc244fd8'\nSOURCE_REV = 'e706e27acc88070918308a10922b8f7174d0825e'\nSAMPLES = {\n    '01_walk': 'Walk forward.',\n    '02_turn': 'Turn left.',\n    '03_wave': 'Wave your right hand.',\n    '04_bow': 'Bow down and stand up.',\n    '05_stop': 'Stop.',\n}\nTEXT_PROMPTS = {\n    'walk': 'A person walks straight ahead.',\n    'wave': 'A person waves with their right hand.',\n    'bow': 'A person bows down and then stands upright.',\n}\n\n\ndef prepare_audio(work):\n    audio = work / 'audio'\n    audio.mkdir(exist_ok=True)\n    if all((audio / f'{name}.wav').is_file() for name in SAMPLES):\n        return audio\n    voice = shutil.which('espeak-ng') or shutil.which('espeak')\n    if voice is None:\n        subprocess.run(['apt-get', 'update', '-qq'], check=True)\n        subprocess.run(['apt-get', 'install', '-y', '-qq', 'espeak-ng'], check=True)\n        voice = shutil.which('espeak-ng')\n    for name, words in SAMPLES.items():\n        path = audio / f'{name}.wav'\n        if not path.exists():\n            subprocess.run([voice, '-v', 'en-us', '-s', '150', '-w', str(path), words], check=True)\n    return audio\n\n\ndef transcribe(work):\n    audio = prepare_audio(work)\n    asr_path = work / 'whisper-tiny-en'\n    start = time.perf_counter()\n    model = WhisperModel(str(asr_path), device='cpu', compute_type='int8', cpu_threads=4)\n    result = {\n        'model': 'Systran/faster-whisper-tiny.en',\n        'device': 'cpu',\n        'compute_type': 'int8',\n        'model_load_seconds': time.perf_counter() - start,\n        'samples': [],\n    }\n    for name, words in SAMPLES.items():\n        start = time.perf_counter()\n        wav = audio / f'{name}.wav'\n        decoded = subprocess.check_output([\n            'ffmpeg', '-v', 'error', '-i', str(wav), '-ac', '1', '-ar', '16000', '-f', 'f32le', '-'\n        ])\n        samples = np.frombuffer(decoded, dtype='<f4').copy()\n        segments, info = model.transcribe(\n            samples, language='en', beam_size=1, vad_filter=False, condition_on_previous_text=False\n        )\n        transcript = ' '.join(segment.text.strip() for segment in segments).strip()\n        result['samples'].append({\n            'file': wav.name,\n            'reference': words,\n            'transcript': transcript,\n            'seconds': time.perf_counter() - start,\n            'audio_seconds': float(info.duration),\n        })\n    return result\n\n\ndef intent(text):\n    words = set(re.findall(r'[a-z]+', text.lower()))\n    if 'stop' in words:\n        return 'stop'\n    if 'left' in words and ('turn' in words or 'left' == text.lower().strip('. ')):\n        return 'left'\n    if 'wave' in words:\n        return 'wave'\n    if 'bow' in words:\n        return 'bow'\n    if 'walk' in words or 'forward' in words:\n        return 'walk'\n    return None\n\n\ndef path_target(command, start, frames, fps):\n    seconds = np.arange(1, frames + 1, dtype=np.float32) / fps\n    if command == 'stop':\n        displacement = np.zeros((frames, 2), np.float32)\n    else:\n        turn = np.zeros(frames, np.float32) if command == 'walk' else np.minimum(seconds / 1.2, 1) * (np.pi / 2)\n        step = (0.65 / fps) * np.stack((-np.sin(turn), np.cos(turn)), axis=-1)\n        displacement = np.cumsum(step, axis=0)\n    return (start + displacement).astype(np.float32)\n\n\ndef sync(device):\n    if device == 'cuda':\n        torch.cuda.synchronize()\n\n\ndef load_features(commands, device):\n    os.environ['TEXT_ENCODER_DEVICE'] = device\n    start = time.perf_counter()\n    encoder = load_text_encoder(mode='local', device=device)\n    sync(device)\n    load_seconds = time.perf_counter() - start\n    features, elapsed = {}, {}\n    for command in dict.fromkeys(commands):\n        start = time.perf_counter()\n        inputs, lengths = encoder([command])\n        features[command] = (inputs.detach().float().cpu(), int(lengths[0]))\n        sync(device)\n        elapsed[command] = time.perf_counter() - start\n    del encoder\n    gc.collect()\n    if device == 'cuda':\n        torch.cuda.empty_cache()\n    return features, load_seconds, elapsed\n\n\n@torch.inference_mode()\ndef play(model, events, mode, steps, device, features):\n    pieces, windows, constraints = [], [], []\n    history = None\n    horizon = model.gen_horizon_len\n    fps = float(model.motion_rep.fps)\n    root_start = np.zeros(2, np.float32)\n    max_history = int(fps * 4)\n    for event in events:\n        for _ in range(event['chunks']):\n            history_size = 0 if history is None else history.shape[1]\n            mask = observed = None\n            if mode == 'path':\n                target = path_target(event['intent'], root_start, horizon, fps)\n                keyframes = torch.arange(history_size, history_size + horizon, device=device)\n                root2d = torch.from_numpy(target).to(device)\n                cond = Root2DConstraintSet(model.skeleton, keyframes, root2d)\n                length = torch.tensor([history_size + horizon], device=device)\n                observed, mask = model.motion_rep.create_conditions_from_constraints_batched(\n                    [cond], length, to_normalize=True, device=device\n                )\n                feature, length_text = torch.zeros((1, 1, 4096)), 0\n                constraints.append(target)\n            else:\n                feature, length_text = features[event['command']]\n            feature = feature.to(device)\n            text_mask = (torch.arange(feature.shape[1], device=device) < length_text).unsqueeze(0)\n            sync(device)\n            start_time = time.perf_counter()\n            window = model.autoregressive_step(\n                num_frames=history_size + horizon,\n                num_denoising_steps=steps,\n                motion_mask=mask,\n                observed_motion=observed,\n                cfg_weight=(2.0, 2.0),\n                text_feat=feature,\n                text_pad_mask=text_mask,\n                init_history_sequence=history,\n            )\n            sync(device)\n            elapsed = time.perf_counter() - start_time\n            latest = window[:, -horizon:]\n            inverse = model.motion_rep.inverse(latest, is_normalized=True)\n            root = inverse['root_positions'][0, :, [0, 2]].float().cpu().numpy()\n            if root.shape != (horizon, 2):\n                raise ValueError(f'Unexpected root ground-plane shape {root.shape}')\n            root_start = root[-1]\n            err = np.linalg.norm(root - target, axis=-1) if mode == 'path' else None\n            windows.append({\n                'frame': len(pieces) * horizon,\n                'history_frames': history_size,\n                'generation_s': elapsed,\n                'constraint_mean_m': None if err is None else float(err.mean()),\n                'constraint_final_m': None if err is None else float(err[-1]),\n            })\n            pieces.append(latest)\n            history = window[:, -max_history:]\n    motion = model.motion_rep.inverse(torch.cat(pieces, dim=1), is_normalized=True)\n    out = {key: value[0].float().cpu().numpy() for key, value in motion.items() if torch.is_tensor(value)}\n    out['parents'] = model.skeleton.joint_parents.cpu().numpy()\n    out['joint_names'] = np.asarray(model.skeleton.bone_order_names)\n    out['fps'] = np.asarray(fps)\n    if mode == 'path':\n        out['target_root_xz'] = np.concatenate(constraints)\n    return out, windows\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument('--work', type=Path, default=Path('/content/ardy-repro'))\n    p.add_argument('--device', choices=['cpu', 'cuda'], default='cuda')\n    p.add_argument('--mode', choices=['path', 'text'], default='path')\n    p.add_argument('--input', choices=['voice', 'reference'], default='voice')\n    p.add_argument('--encoder-device', choices=['cpu', 'cuda'], default='cpu')\n    p.add_argument('--steps', type=int, default=4)\n    p.add_argument('--seed', type=int, default=20261121)\n    args = p.parse_args()\n    args.work.mkdir(exist_ok=True, parents=True)\n    stat = args.work / 'results'\n    stat.mkdir(exist_ok=True)\n    torch.set_num_threads(4)\n\n    if args.input == 'voice':\n        asr = transcribe(args.work)\n        (stat / 'asr.json').write_text(json.dumps(asr, indent=2) + '\\n')\n        heard = {Path(row['file']).stem: row['transcript'] for row in asr['samples']}\n        print('ASR:', json.dumps(asr, ensure_ascii=False), flush=True)\n    else:\n        heard = SAMPLES\n    names = ('01_walk', '02_turn', '05_stop') if args.mode == 'path' else ('01_walk', '03_wave', '04_bow')\n    events = []\n    for name in names:\n        transcript = heard[name]\n        kind = intent(transcript)\n        if args.mode == 'path' and kind not in {'walk', 'left', 'stop'}:\n            raise ValueError(f'Unsupported locomotion input {name}: {transcript!r}; no motion fabricated')\n        command = transcript\n        if args.mode == 'text':\n            command = TEXT_PROMPTS.get(kind, transcript)\n            if not command.strip():\n                raise ValueError(f'Empty speech transcription for {name}')\n        events.append({'file': f'{name}.wav', 'command': command, 'transcript': transcript,\n                       'intent': kind, 'chunks': 1 if args.mode == 'path' else 2})\n    features, encoder_load_s, encoder_seconds = {}, 0.0, {}\n    if args.mode == 'text':\n        features, encoder_load_s, encoder_seconds = load_features(\n            [event['command'] for event in events], args.encoder_device\n        )\n\n    torch.manual_seed(args.seed)\n    if args.device == 'cuda':\n        torch.cuda.reset_peak_memory_stats()\n    load_start = time.perf_counter()\n    model = load_model(MODEL, device=args.device, checkpoints_dir=args.work / 'weights', text_encoder=False)\n    sync(args.device)\n    load_elapsed = time.perf_counter() - load_start\n    if not 1 <= args.steps <= int(model.diffusion.num_base_steps):\n        raise ValueError('Invalid diffusion steps for checkpoint')\n    next_frame = 0\n    for event in events:\n        event['frame'] = next_frame\n        next_frame += event['chunks'] * model.gen_horizon_len\n    motion, windows = play(model, events, args.mode, args.steps, args.device, features)\n    joints = motion['posed_joints']\n    if joints.shape != (next_frame, 27, 3) or not all(np.isfinite(v).all() for v in motion.values() if v.dtype.kind in 'fc'):\n        raise ValueError(f'Invalid generated motion {joints.shape}')\n    root = motion['root_positions'][:, [0, 2]]\n    jump = np.linalg.norm(np.diff(joints, axis=0), axis=-1)\n    boundaries = [\n        {'frame': e['frame'], 'mean_joint_step_m': float(jump[e['frame'] - 1].mean())} for e in events[1:]\n    ]\n    timings = [item['generation_s'] for item in windows]\n    summary = {\n        'time_cn': datetime.now(timezone(timedelta(hours=8))).isoformat(timespec='seconds'),\n        'upstream_commit': SOURCE_REV,\n        'checkpoint': MODEL,\n        'checkpoint_revision': CHECKPOINT_REV,\n        'mode': args.mode,\n        'input': args.input,\n        'device': torch.cuda.get_device_name() if args.device == 'cuda' else platform.processor() or 'CPU',\n        'torch': torch.__version__,\n        'numpy': np.__version__,\n        'seed': args.seed,\n        'diffusion_steps': args.steps,\n        'fps': float(motion['fps']),\n        'horizon_frames': int(model.gen_horizon_len),\n        'shape': list(joints.shape),\n        'events': events,\n        'load_s': load_elapsed,\n        'encoder_load_s': encoder_load_s,\n        'encoder_seconds': encoder_seconds,\n        'windows': windows,\n        'median_generation_ms': 1000 * statistics.median(timings),\n        'median_after_first_ms': 1000 * statistics.median(timings[1:]),\n        'max_cuda_mib': torch.cuda.max_memory_allocated() / 2**20 if args.device == 'cuda' else None,\n        'max_reserved_cuda_mib': torch.cuda.max_memory_reserved() / 2**20 if args.device == 'cuda' else None,\n        'boundaries': boundaries,\n        'median_joint_step_m': float(np.median(jump.mean(axis=1))),\n        'root_start_xz_m': root[0].tolist(),\n        'root_end_xz_m': root[-1].tolist(),\n        'root_path_m': float(np.linalg.norm(np.diff(root, axis=0), axis=1).sum()),\n    }\n    stem = f'{args.mode}_{args.input}_{args.steps}step'\n    np.savez_compressed(stat / f'{stem}.npz', **motion)\n    (stat / f'{stem}.json').write_text(json.dumps(summary, indent=2) + '\\n')\n    print('MOTION_METRICS:', json.dumps(summary, ensure_ascii=False), flush=True)\n\n\nif __name__ == '__main__':\n    main()\n", encoding='utf-8')
from pathlib import Path
root = Path('/content/ardy-repro')
root.mkdir(exist_ok=True)
(root / "render_motion.py").write_text("\"\"\"Render official ARDY output with a fixed camera and ground-plane track.\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\nimport subprocess\nimport tempfile\n\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nfrom matplotlib.animation import FFMpegWriter, FuncAnimation\nfrom mpl_toolkits.mplot3d.art3d import Line3DCollection\nimport numpy as np\n\n\ndef render(npz_path, output, audio_dir=None):\n    with np.load(npz_path, allow_pickle=False) as data:\n        joints = data['posed_joints']\n        parents = data['parents']\n        root = data['root_positions'][:, [0, 2]]\n        target = data['target_root_xz'] if 'target_root_xz' in data else None\n        fps = float(data['fps'])\n    metrics = json.loads(npz_path.with_suffix('.json').read_text())\n    events = metrics['events']\n    total = len(joints)\n    assert joints.shape == (total, 27, 3) and parents.shape == (27,)\n    child = np.flatnonzero(parents >= 0)\n    plot_joints = joints[:, :, [0, 2, 1]]\n    floor = float(np.min(plot_joints[:, :, 2]))\n    low = plot_joints.reshape(-1, 3).min(axis=0) - np.array([.2, .2, .0])\n    high = plot_joints.reshape(-1, 3).max(axis=0) + np.array([.2, .2, .15])\n    fig = plt.figure(figsize=(10, 5), facecolor='white')\n    ax = fig.add_subplot(121, projection='3d')\n    topo = fig.add_subplot(122)\n    ax.set(xlim=(low[0], high[0]), ylim=(low[1], high[1]), zlim=(floor, high[2]))\n    ax.set_box_aspect((high[0]-low[0], high[1]-low[1], high[2]-floor))\n    ax.view_init(18, -58)\n    ax.set(xlabel='X (m)', ylabel='Z (m)', zlabel='Height Y (m)', title='Generated skeleton')\n    first = plot_joints[0]\n    bones = np.stack((first[child], first[parents[child]]), axis=1)\n    skeleton = Line3DCollection(bones, colors='#245880', linewidths=2.8)\n    ax.add_collection3d(skeleton, autolim=False)\n    xy = np.concatenate((root, target)) if target is not None else root\n    margin = 0.25\n    topo.set(xlim=(xy[:,0].min()-margin, xy[:,0].max()+margin),\n             ylim=(xy[:,1].min()-margin, xy[:,1].max()+margin),\n             xlabel='X (m)', ylabel='Z (m)', title='Root path')\n    topo.set_aspect('equal')\n    topo.grid(alpha=.2)\n    if target is not None:\n        topo.plot(target[:,0], target[:,1], '--', lw=2, color='#238360', label='Commanded path')\n    actual, = topo.plot([], [], color='#245880', lw=2, label='Generated motion')\n    point, = topo.plot([], [], 'o', color='#245880', markersize=5)\n    topo.legend(loc='upper left', fontsize=8)\n    label = fig.suptitle('')\n    fig.subplots_adjust(wspace=.25, top=.86, bottom=.14)\n\n    def draw(index):\n        pose = plot_joints[index]\n        skeleton.set_segments(np.stack((pose[child], pose[parents[child]]), axis=1))\n        actual.set_data(root[:index+1,0], root[:index+1,1])\n        point.set_data([root[index,0]], [root[index,1]])\n        event = next(item for item in reversed(events) if item['frame'] <= index)\n        label.set_text(f\"ARDY on {metrics['device']}   |   {index/fps:.2f} s   |   {event['command']}\")\n        return skeleton, actual, point, label\n\n    animation = FuncAnimation(fig, draw, frames=total, cache_frame_data=False)\n    with tempfile.TemporaryDirectory(dir=output.parent, prefix='render-') as directory:\n        silent = Path(directory) / 'silent.mp4'\n        animation.save(silent, writer=FFMpegWriter(fps=fps, codec='libx264',\n                       extra_args=['-pix_fmt','yuv420p','-preset','veryfast']), dpi=116)\n        if audio_dir is None:\n            silent.replace(output)\n        else:\n            inputs = ['ffmpeg', '-y', '-v', 'error', '-i', str(silent)]\n            filters = []\n            for j, event in enumerate(events, 1):\n                inputs.extend(['-i',str(audio_dir/event['file'])])\n                delay = int(round(event['frame']/fps*1000))\n                filters.append(f'[{j}:a]adelay={delay}:all=1[a{j}]')\n            mix = ''.join(f'[a{j}]' for j in range(1,len(events)+1))\n            filters.append(f'{mix}amix=inputs={len(events)}:normalize=0,apad[a]')\n            subprocess.run(inputs + ['-filter_complex', ';'.join(filters), '-map', '0:v',\n                           '-map','[a]', '-c:v','copy', '-c:a','aac', '-shortest', str(output)], check=True)\n    plt.close(fig)\n    print('Rendered', output.name, output.stat().st_size, 'bytes', flush=True)\n\n\nif __name__ == '__main__':\n    p = argparse.ArgumentParser()\n    p.add_argument('npz', type=Path)\n    p.add_argument('output', type=Path)\n    p.add_argument('--audio-dir', type=Path)\n    a = p.parse_args()\n    render(a.npz, a.output, a.audio_dir)\n", encoding='utf-8')
import subprocess
from pathlib import Path
root = Path('/content/ardy-repro')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/"run_ardy.py"), *["--mode","path","--input","voice","--device","cuda"]], cwd=root, check=True)


In [ ]:
import subprocess
from pathlib import Path
root = Path('/content/ardy-repro')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/"run_ardy.py"), *["--mode","path","--input","reference","--device","cuda"]], cwd=root, check=True)
import subprocess
from pathlib import Path
root = Path('/content/ardy-repro')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/"run_ardy.py"), *["--mode","path","--input","voice","--device","cuda","--steps","10"]], cwd=root, check=True)
import subprocess
from pathlib import Path
root = Path('/content/ardy-repro')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/"render_motion.py"), *["results/path_voice_4step.npz","results/ARDY_voice_path_T4.mp4","--audio-dir","audio"]], cwd=root, check=True)
from IPython.display import Video
Video('/content/ardy-repro/results/ARDY_voice_path_T4.mp4', embed=True)


## G1 口令打断和噪声实验


In [ ]:
from pathlib import Path
root = Path('/content/ardy-repro')
root.mkdir(exist_ok=True)
(root / "prepare_g1.py").write_text("import concurrent.futures\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nfrom huggingface_hub import HfApi, snapshot_download\n\nbase = Path('/content/ardy-repro')\nos.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'\nmodels = ['ARDY-G1-RP-25FPS-Horizon8', 'ARDY-G1-RP-25FPS-Horizon52']\napi = HfApi()\ndef model_files(name):\n    repo = 'nvidia/' + name\n    revision = {'ARDY-G1-RP-25FPS-Horizon8': 'fb815e8021d700af523ac32d9851b7e489fbb60b', 'ARDY-G1-RP-25FPS-Horizon52': '3d4c1ff16af4871b18125c95271e77c7f6f530d7'}[name]\n    target = base / 'weights' / name\n    snapshot_download(repo, revision=revision, allow_patterns=['config.yaml','stats/*/*.npy','denoiser.safetensors','tokenizer.safetensors','LICENSE'], local_dir=target, token=False, max_workers=3)\n    print(name, revision, sum(x.stat().st_size for x in target.rglob('*.safetensors')), 'B', flush=True)\n    return name, revision\n\nwith concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:\n    sim = ex.submit(subprocess.run, ['uv','pip','install','--python',str(base/'python/bin/python'),'mujoco==4.0.1'], check=True)\n    revisions = dict(ex.map(model_files,models))\n    sim.result()\n(base / 'g1_weight_revisions.json').write_text(json.dumps(revisions, indent=2) + '\\n')\n", encoding='utf-8')
import subprocess
from pathlib import Path
root = Path('/content/ardy-repro')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/"prepare_g1.py"), *[]], cwd=root, check=True)
from pathlib import Path
root = Path('/content/ardy-repro')
root.mkdir(exist_ok=True)
(root / "advanced_g1.py").write_text("\"\"\"Causal timestamped replay of three fixed voice control sequences on ARDY G1.\"\"\"\n\nimport csv\nimport gc\nimport json\nfrom pathlib import Path\nimport re\nimport subprocess\nimport time\n\nimport numpy as np\nimport torch\nfrom ardy.constraints import Root2DConstraintSet\nfrom ardy.model.load_model import load_model\nfrom faster_whisper import WhisperModel\n\nfrom run_ardy import prepare_audio, SAMPLES, SOURCE_REV\n\n\nWORK = Path('/content/ardy-repro')\nRESULTS = WORK / 'results'\nMODELS = ('ARDY-G1-RP-25FPS-Horizon8', 'ARDY-G1-RP-25FPS-Horizon52')\nSCENARIOS = (\n    ('before_boundary', 1.00, 2.95),\n    ('after_boundary', 1.32, 3.62),\n    ('rapid_override', 1.52, 2.30),\n)\nCOMMANDS = {'walk forward': 'walk', 'turn left': 'left', 'stop': 'stop'}\nSEEDS = (20261121, 20261122, 20261123)\nDEVICE = 'cuda'\nTEST_SECONDS = 9.6\nSTEPS = 4\nHISTORY = 100\n\n\ndef normalize(text):\n    return ' '.join(re.findall(r'[a-z]+', text.lower()))\n\n\ndef noisy_asr():\n    audio = prepare_audio(WORK)\n    espeak = 'espeak-ng'\n    negative = audio / '06_do_not_stop.wav'\n    subprocess.run([espeak, '-v','en-us', '-s','150', '-w',str(negative), 'Do not stop.'], check=True)\n    examples = {**SAMPLES, '06_do_not_stop': 'Do not stop.'}\n    model = WhisperModel(str(WORK / 'whisper-tiny-en'), device='cpu', compute_type='int8', cpu_threads=4)\n    rng = np.random.default_rng(20261122)\n    records = []\n    for name, phrase in examples.items():\n        data = subprocess.check_output([\n            'ffmpeg','-v','error','-i',str(audio / f'{name}.wav'),'-ac','1','-ar','16000','-f','f32le','-'\n        ])\n        clean = np.frombuffer(data, dtype='<f4').copy()\n        base_noise = rng.normal(size=len(clean)).astype('float32')\n        rms = np.sqrt(np.mean(np.square(clean)))\n        for snr in (None, 10, 0):\n            wave = clean\n            if snr is not None:\n                noise_rms = np.sqrt(np.mean(np.square(base_noise)))\n                wave = np.clip(clean + base_noise * rms / (noise_rms * 10 ** (snr / 20)), -1, 1)\n            start = time.perf_counter()\n            segments, _ = model.transcribe(\n                wave, language='en', beam_size=1, vad_filter=False, condition_on_previous_text=False\n            )\n            recognized = ' '.join(seg.text.strip() for seg in segments).strip()\n            seconds = time.perf_counter() - start\n            records.append({\n                'file':name+'.wav','reference':phrase,'transcript':recognized,'snr_db':snr,\n                'decoder_s':seconds,'exact':normalize(recognized)==normalize(phrase),\n                'expected_intent':COMMANDS.get(normalize(phrase)),\n                'applied_intent':COMMANDS.get(normalize(recognized)),\n            })\n    return records\n\n\ndef events_for(scenario, clean, fps):\n    name, left_end, stop_end = scenario\n    chosen = [clean['01_walk.wav'], clean['02_turn.wav'], clean['05_stop.wav']]\n    return [\n        {'intent':item['applied_intent'], 'due_intent':item['expected_intent'], 'speech_end_s':end,\n         'recognition_s':item['decoder_s'], 'ready_s':max(0, end+item['decoder_s']),\n         'ready_frame':max(0,int(np.ceil((end+item['decoder_s'])*fps))),\n         'status':'pending','file':item['file']}\n        for item, end in zip(chosen, (-2.,left_end,stop_end))\n    ]\n\n\ndef advance_target(pos, action, start, applied, horizon, fps):\n    seconds = (np.arange(start, start+horizon, dtype=np.float32)-applied+1)/fps\n    if action == 'stop':\n        return np.repeat(pos[None], horizon, axis=0)\n    angles = np.minimum(seconds/1.2,1)*np.pi/2 if action == 'left' else np.zeros_like(seconds)\n    speed = 0.55\n    increments = speed/fps * np.stack((-np.sin(angles), np.cos(angles)),axis=-1)\n    return (pos + np.cumsum(increments,axis=0)).astype('float32')\n\n\n@torch.inference_mode()\ndef rollout(model, scenario, clean, seed, save_path=None):\n    fps = float(model.motion_rep.fps)\n    horizon = model.gen_horizon_len\n    count = int(np.round(TEST_SECONDS*fps))\n    torch.manual_seed(seed)\n    events = events_for(scenario,clean,fps)\n    assert events[0]['intent'] == 'walk' and all(event['intent']==event['due_intent'] for event in events)\n    text = torch.zeros((1,1,4096),device=DEVICE)\n    text_mask = torch.zeros((1,1),dtype=torch.bool,device=DEVICE)\n    retained = None\n    chunks, trajectory, latencies = [], [], []\n    start_xz = np.zeros(2,np.float32)\n    state, changed_at = 'walk', 0\n    for frame in range(0,count,horizon):\n        ready = [e for e in events if e['status']=='pending' and e['ready_frame']<=frame]\n        if ready:\n            for event in ready[:-1]:\n                event['status']='superseded'\n            state, changed_at = ready[-1]['intent'], frame\n            ready[-1]['status']='applied'\n            ready[-1]['frame']=frame\n            ready[-1]['command']=state\n            ready[-1]['queue_s']=frame/fps-ready[-1]['ready_s']\n\n        end = start_xz if not trajectory else trajectory[-1][-1]\n        target = advance_target(end,state,frame,changed_at,horizon,fps)\n        trajectory.append(target)\n        nh = 0 if retained is None else retained.shape[1]\n        indices = torch.arange(nh,nh+horizon,device=DEVICE)\n        cond = Root2DConstraintSet(model.skeleton,indices,torch.from_numpy(target).to(DEVICE))\n        length = torch.tensor([nh+horizon],device=DEVICE)\n        observed, mask = model.motion_rep.create_conditions_from_constraints_batched(\n            [cond],length,to_normalize=True,device=DEVICE\n        )\n        torch.cuda.synchronize()\n        tick = time.perf_counter()\n        m = model.autoregressive_step(num_frames=nh+horizon,num_denoising_steps=STEPS,\n            motion_mask=mask,observed_motion=observed,cfg_weight=(2.,2.),\n            text_feat=text,text_pad_mask=text_mask,init_history_sequence=retained)\n        torch.cuda.synchronize()\n        elapsed = time.perf_counter()-tick\n        latencies.append(elapsed)\n        if ready:\n            ready[-1]['first_chunk_generation_s']=elapsed\n            ready[-1]['estimated_aud_end_to_motion_ready_s']=frame/fps-ready[-1]['speech_end_s']+elapsed\n        chunks.append(m[:,-horizon:])\n        retained = m[:,-HISTORY:]\n    generated = model.motion_rep.inverse(torch.cat(chunks,dim=1)[:,:count],is_normalized=True)\n    out = {key: value[0].float().cpu().numpy() for key,value in generated.items() if torch.is_tensor(value)}\n    joints = out['posed_joints']\n    assert joints.shape == (count,34,3) and np.isfinite(joints).all()\n    root = out['root_positions'][:,[0,2]]\n    targets = np.concatenate(trajectory)[:count]\n    error = np.linalg.norm(root-targets,axis=-1)\n    stop = events[-1]\n    stop_ready = stop['ready_frame']\n    stop_applied = stop['frame']\n    distance = np.linalg.norm(np.diff(root,axis=0),axis=1)\n    base_index = max(0,stop_ready-1)\n    stop_pending = float(distance[base_index:max(base_index,stop_applied-1)].sum())\n    stationary = np.linalg.norm(root[stop_applied:]-targets[stop_applied],axis=1)\n    result = {\n        'scenario':scenario[0],'seed':seed,'events':events,'horizon_frames':horizon,'fps':fps,\n        'generated_windows':len(chunks),'joints_shape':list(joints.shape),\n        'mean_root_path_error_m':float(error.mean()),\n        'stop_ready_to_apply_path_m':stop_pending,\n        'stop_ready_to_end_net_displacement_m':float(np.linalg.norm(root[-1]-root[base_index])),\n        'post_stop_mean_target_error_m':float(stationary.mean()),\n        'max_root_speed_m_s':float(distance.max()*fps),\n        'median_window_generation_s':float(np.median(latencies[1:])),\n        'actual_generation_s':float(sum(latencies)),\n        'cuda_peak_mib':torch.cuda.max_memory_allocated()/2**20,\n        'computed_playback_s':count/fps,\n    }\n    if save_path is not None:\n        out.update(fps=np.asarray(fps),\n            parents=model.skeleton.joint_parents.cpu().numpy(),\n            joint_names=np.asarray(model.skeleton.bone_order_names),target_root_xz=targets)\n        np.savez_compressed(save_path,**out)\n        result['device']=torch.cuda.get_device_name()\n        play_events=[event for event in events if event['status']=='applied']\n        save_path.with_suffix('.json').write_text(json.dumps({**result,'events':play_events},indent=2)+'\\n')\n        try:\n            from ardy.exports.mujoco import MujocoQposConverter\n            qpos = MujocoQposConverter(model.skeleton).dict_to_qpos(out, device=DEVICE)\n            if hasattr(qpos, 'cpu'):\n                qpos=qpos.detach().cpu().numpy()\n            if qpos.shape != (count,36) or not np.isfinite(qpos).all():\n                raise ValueError(f'Invalid qpos shape {qpos.shape}')\n            np.savetxt(save_path.with_suffix('.csv'),qpos,delimiter=',',fmt='%.8f')\n            result['qpos_shape']=list(qpos.shape)\n        except (TypeError, ValueError, AttributeError, IndexError, RuntimeError) as error:\n            result['qpos_export_error']=repr(error)\n            print('QPOS export needs follow-up:',type(error).__name__,str(error),flush=True)\n    return result\n\n\ndef main():\n    RESULTS.mkdir(exist_ok=True)\n    torch.set_num_threads(4)\n    records = noisy_asr()\n    (RESULTS/'advanced_asr_noise.json').write_text(json.dumps(records,indent=2)+'\\n')\n    print('ADV_ASR:',json.dumps(records),flush=True)\n    clean = {r['file']:r for r in records if r['snr_db'] is None}\n    rows=[]\n    revisions=json.loads((WORK/'g1_weight_revisions.json').read_text())\n    for name in MODELS:\n        torch.cuda.reset_peak_memory_stats()\n        model=load_model(name,device=DEVICE,checkpoints_dir=WORK/'weights',text_encoder=False)\n        for scenario in SCENARIOS:\n            for seed in SEEDS:\n                path = RESULTS / f'G1_H{model.gen_horizon_len}_{scenario[0]}.npz' if seed == SEEDS[0] else None\n                result = rollout(model,scenario,clean,seed,path)\n                result.update(model=name,revision=revisions[name])\n                rows.append(result)\n                print('G1_RUN:',name,scenario[0],seed,\n                      [(e['intent'],e['status'],e.get('queue_s')) for e in result['events']],\n                      round(result['mean_root_path_error_m'],4),flush=True)\n        del model\n        gc.collect()\n        torch.cuda.empty_cache()\n    (RESULTS/'advanced_g1_replay.json').write_text(json.dumps({\n        'upstream_commit':SOURCE_REV,'device':torch.cuda.get_device_name(),'diffusion_steps':STEPS,\n        'history_frames':HISTORY,'seeds':SEEDS,'scenarios':SCENARIOS,'rows':rows},indent=2)+'\\n')\n\n    for name in MODELS:\n        group=[r for r in rows if r['model']==name]\n        samples=[r for r in group if r['seed']==SEEDS[0]]\n        applied=[e for r in samples for e in r['events'][1:] if e['status']=='applied']\n        print('G1_SUMMARY:',json.dumps({\n            'model':name,'runs':len(group),\n            'applied':len(applied),'spoken_dynamic_events':len(samples)*2,\n            'mean_queue_s':float(np.mean([e['queue_s'] for e in applied])),\n            'max_queue_s':max(e['queue_s'] for e in applied),\n            'superseded_events':[(r['scenario'],e['intent']) for r in samples for e in r['events'] if e['status']=='superseded'],\n            'mean_path_error_cm':100*float(np.mean([r['mean_root_path_error_m'] for r in group])),\n            'mean_stop_pending_distance_m':float(np.mean([r['stop_ready_to_apply_path_m'] for r in group])),\n            'mean_post_stop_target_error_cm':100*float(np.mean([r['post_stop_mean_target_error_m'] for r in group])),\n            'representative_window_ms':1000*float(np.median([r['median_window_generation_s'] for r in group])),\n            'peak_allocated_mib':max(r['cuda_peak_mib'] for r in group),\n            'all_qpos_shape':[r.get('qpos_shape') for r in samples],\n        }),flush=True)\n\n\nif __name__ == '__main__':\n    main()\n", encoding='utf-8')
import subprocess
from pathlib import Path
root = Path('/content/ardy-repro')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/"advanced_g1.py"), *[]], cwd=root, check=True)


In [ ]:
from pathlib import Path
root = Path('/content/ardy-repro')
root.mkdir(exist_ok=True)
(root / "replan_g1.py").write_text("\"\"\"Causal timestamped replay of three fixed voice control sequences on ARDY G1.\"\"\"\n\nimport csv\nimport gc\nimport json\nfrom pathlib import Path\nimport re\nimport subprocess\nimport time\n\nimport numpy as np\nimport torch\nfrom ardy.constraints import Root2DConstraintSet\nfrom ardy.model.load_model import load_model\nfrom faster_whisper import WhisperModel\n\nfrom run_ardy import prepare_audio, SAMPLES, SOURCE_REV\n\n\nWORK = Path('/content/ardy-repro')\nRESULTS = WORK / 'results'\nMODELS = ('ARDY-G1-RP-25FPS-Horizon8', 'ARDY-G1-RP-25FPS-Horizon52')\nSCENARIOS = (\n    ('before_boundary', 1.00, 2.95),\n    ('after_boundary', 1.32, 3.62),\n    ('rapid_override', 1.52, 2.30),\n)\nCOMMANDS = {'walk forward': 'walk', 'turn left': 'left', 'stop': 'stop'}\nSEEDS = (20261121, 20261122, 20261123)\nDEVICE = 'cuda'\nTEST_SECONDS = 9.6\nSTEPS = 4\nHISTORY = 100\n\n\ndef normalize(text):\n    return ' '.join(re.findall(r'[a-z]+', text.lower()))\n\n\ndef noisy_asr():\n    audio = prepare_audio(WORK)\n    espeak = 'espeak-ng'\n    negative = audio / '06_do_not_stop.wav'\n    subprocess.run([espeak, '-v','en-us', '-s','150', '-w',str(negative), 'Do not stop.'], check=True)\n    examples = {**SAMPLES, '06_do_not_stop': 'Do not stop.'}\n    model = WhisperModel(str(WORK / 'whisper-tiny-en'), device='cpu', compute_type='int8', cpu_threads=4)\n    rng = np.random.default_rng(20261122)\n    records = []\n    for name, phrase in examples.items():\n        data = subprocess.check_output([\n            'ffmpeg','-v','error','-i',str(audio / f'{name}.wav'),'-ac','1','-ar','16000','-f','f32le','-'\n        ])\n        clean = np.frombuffer(data, dtype='<f4').copy()\n        base_noise = rng.normal(size=len(clean)).astype('float32')\n        rms = np.sqrt(np.mean(np.square(clean)))\n        for snr in (None, 10, 0):\n            wave = clean\n            if snr is not None:\n                noise_rms = np.sqrt(np.mean(np.square(base_noise)))\n                wave = np.clip(clean + base_noise * rms / (noise_rms * 10 ** (snr / 20)), -1, 1)\n            start = time.perf_counter()\n            segments, _ = model.transcribe(\n                wave, language='en', beam_size=1, vad_filter=False, condition_on_previous_text=False\n            )\n            recognized = ' '.join(seg.text.strip() for seg in segments).strip()\n            seconds = time.perf_counter() - start\n            records.append({\n                'file':name+'.wav','reference':phrase,'transcript':recognized,'snr_db':snr,\n                'decoder_s':seconds,'exact':normalize(recognized)==normalize(phrase),\n                'expected_intent':COMMANDS.get(normalize(phrase)),\n                'applied_intent':COMMANDS.get(normalize(recognized)),\n            })\n    return records\n\n\ndef events_for(scenario, clean, fps):\n    name, left_end, stop_end = scenario\n    chosen = [clean['01_walk.wav'], clean['02_turn.wav'], clean['05_stop.wav']]\n    return [\n        {'intent':item['applied_intent'], 'due_intent':item['expected_intent'], 'speech_end_s':end,\n         'recognition_s':item['decoder_s'], 'ready_s':max(0, end+item['decoder_s']),\n         'ready_frame':max(0,int(np.ceil((end+item['decoder_s'])*fps))),\n         'status':'pending','file':item['file']}\n        for item, end in zip(chosen, (-2.,left_end,stop_end))\n    ]\n\n\ndef advance_target(pos, action, start, applied, horizon, fps):\n    seconds = (np.arange(start, start+horizon, dtype=np.float32)-applied+1)/fps\n    if action == 'stop':\n        return np.repeat(pos[None], horizon, axis=0)\n    angles = np.minimum(seconds/1.2,1)*np.pi/2 if action == 'left' else np.zeros_like(seconds)\n    speed = 0.55\n    increments = speed/fps * np.stack((-np.sin(angles), np.cos(angles)),axis=-1)\n    return (pos + np.cumsum(increments,axis=0)).astype('float32')\n\n\n@torch.inference_mode()\ndef rollout(model, scenario, clean, seed, save_path=None, policy='whole'):\n    fps = float(model.motion_rep.fps)\n    horizon = model.gen_horizon_len\n    count = int(np.round(TEST_SECONDS*fps))\n    torch.manual_seed(seed)\n    events = events_for(scenario,clean,fps)\n    assert events[0]['intent'] == 'walk' and all(event['intent']==event['due_intent'] for event in events)\n    text = torch.zeros((1,1,4096),device=DEVICE)\n    text_mask = torch.zeros((1,1),dtype=torch.bool,device=DEVICE)\n    retained = None\n    chunks, trajectory, latencies = [], [], []\n    start_xz = np.zeros(2,np.float32)\n    state, changed_at = 'walk', 0\n    frame = 0\n    while frame < count:\n        ready = [e for e in events if e['status']=='pending' and e['ready_frame']<=frame]\n        if ready:\n            for event in ready[:-1]:\n                event['status']='superseded'\n            state, changed_at = ready[-1]['intent'], frame\n            ready[-1]['status']='applied'\n            ready[-1]['frame']=frame\n            ready[-1]['command']=state\n            ready[-1]['queue_s']=frame/fps-ready[-1]['ready_s']\n\n        end = start_xz if not trajectory else trajectory[-1][-1]\n        lookahead = 52\n        target = advance_target(end,state,frame,changed_at,lookahead,fps)\n        nh = 0 if retained is None else retained.shape[1]\n        indices = torch.arange(nh,nh+lookahead,device=DEVICE)\n        cond = Root2DConstraintSet(model.skeleton,indices,torch.from_numpy(target).to(DEVICE))\n        length = torch.tensor([nh+lookahead],device=DEVICE)\n        observed, mask = model.motion_rep.create_conditions_from_constraints_batched(\n            [cond],length,to_normalize=True,device=DEVICE\n        )\n        torch.cuda.synchronize()\n        tick = time.perf_counter()\n        m = model.autoregressive_step(num_frames=nh+lookahead,num_denoising_steps=STEPS,\n            motion_mask=mask,observed_motion=observed,cfg_weight=(2.,2.),\n            text_feat=text,text_pad_mask=text_mask,init_history_sequence=retained)\n        torch.cuda.synchronize()\n        elapsed = time.perf_counter()-tick\n        latencies.append(elapsed)\n        if ready:\n            ready[-1]['first_chunk_generation_s']=elapsed\n            ready[-1]['estimated_aud_end_to_motion_ready_s']=frame/fps-ready[-1]['speech_end_s']+elapsed\n        commit = min(horizon,count-frame)\n        if policy == 'patch':\n            future = [e['ready_frame'] for e in events if e['status']=='pending' and e['ready_frame']>frame]\n            if future:\n                next_patch = (min(future)+3)//4*4\n                commit = min(commit,next_patch-frame)\n        chunks.append(m[:,nh:nh+commit])\n        trajectory.append(target[:commit])\n        retained = m[:,max(0,nh+commit-HISTORY):nh+commit]\n        frame += commit\n    generated = model.motion_rep.inverse(torch.cat(chunks,dim=1)[:,:count],is_normalized=True)\n    out = {key: value[0].float().cpu().numpy() for key,value in generated.items() if torch.is_tensor(value)}\n    joints = out['posed_joints']\n    assert joints.shape == (count,34,3) and np.isfinite(joints).all()\n    root = out['root_positions'][:,[0,2]]\n    targets = np.concatenate(trajectory)[:count]\n    error = np.linalg.norm(root-targets,axis=-1)\n    stop = events[-1]\n    stop_ready = stop['ready_frame']\n    stop_applied = stop['frame']\n    distance = np.linalg.norm(np.diff(root,axis=0),axis=1)\n    base_index = max(0,stop_ready-1)\n    stop_pending = float(distance[base_index:max(base_index,stop_applied-1)].sum())\n    stationary = np.linalg.norm(root[stop_applied:]-targets[stop_applied],axis=1)\n    result = {\n        'scenario':scenario[0],'seed':seed,'events':events,'horizon_frames':horizon,'fps':fps,\n        'generated_windows':len(chunks),'joints_shape':list(joints.shape),\n        'generated_to_played_ratio':len(chunks)*horizon / count,'lookahead_frames':52,'policy':policy,\n        'mean_root_path_error_m':float(error.mean()),\n        'stop_ready_to_apply_path_m':stop_pending,\n        'stop_ready_to_end_net_displacement_m':float(np.linalg.norm(root[-1]-root[base_index])),\n        'post_stop_mean_target_error_m':float(stationary.mean()),\n        'max_root_speed_m_s':float(distance.max()*fps),\n        'median_window_generation_s':float(np.median(latencies[1:])),\n        'actual_generation_s':float(sum(latencies)),\n        'cuda_peak_mib':torch.cuda.max_memory_allocated()/2**20,\n        'computed_playback_s':count/fps,\n    }\n    if save_path is not None:\n        out.update(fps=np.asarray(fps),\n            parents=model.skeleton.joint_parents.cpu().numpy(),\n            joint_names=np.asarray(model.skeleton.bone_order_names),target_root_xz=targets)\n        np.savez_compressed(save_path,**out)\n        result['device']=torch.cuda.get_device_name()\n        play_events=[event for event in events if event['status']=='applied']\n        save_path.with_suffix('.json').write_text(json.dumps({**result,'events':play_events},indent=2)+'\\n')\n        try:\n            from ardy.exports.mujoco import MujocoQposConverter\n            qpos = MujocoQposConverter(model.skeleton).dict_to_qpos(out, device=DEVICE)\n            if hasattr(qpos, 'cpu'):\n                qpos=qpos.detach().cpu().numpy()\n            if qpos.shape != (count,36) or not np.isfinite(qpos).all():\n                raise ValueError(f'Invalid qpos shape {qpos.shape}')\n            np.savetxt(save_path.with_suffix('.csv'),qpos,delimiter=',',fmt='%.8f')\n            result['qpos_shape']=list(qpos.shape)\n        except (TypeError, ValueError, AttributeError, IndexError, RuntimeError) as error:\n            result['qpos_export_error']=repr(error)\n            print('QPOS export needs follow-up:',type(error).__name__,str(error),flush=True)\n    return result\n\n\ndef main():\n    RESULTS.mkdir(exist_ok=True)\n    torch.set_num_threads(4)\n    records = json.loads((RESULTS/'advanced_asr_noise.json').read_text())\n    clean = {r['file']:r for r in records if r['snr_db'] is None}\n    rows=[]\n    revisions=json.loads((WORK/'g1_weight_revisions.json').read_text())\n    for name in MODELS:\n        policy = 'whole' if name.endswith('8') else 'patch'\n        torch.cuda.reset_peak_memory_stats()\n        model=load_model(name,device=DEVICE,checkpoints_dir=WORK/'weights',text_encoder=False)\n        for scenario in SCENARIOS:\n            for seed in SEEDS:\n                path = RESULTS / f'G1_H{model.gen_horizon_len}_{policy}_52lookahead_{scenario[0]}.npz' if seed == SEEDS[0] else None\n                result = rollout(model,scenario,clean,seed,path,policy)\n                result.update(model=name,revision=revisions[name])\n                rows.append(result)\n                print('G1_RUN:',name,scenario[0],seed,\n                      [(e['intent'],e['status'],e.get('queue_s')) for e in result['events']],\n                      round(result['mean_root_path_error_m'],4),flush=True)\n        del model\n        gc.collect()\n        torch.cuda.empty_cache()\n    (RESULTS/'advanced_g1_replan.json').write_text(json.dumps({\n        'upstream_commit':SOURCE_REV,'device':torch.cuda.get_device_name(),'diffusion_steps':STEPS,\n        'history_frames':HISTORY,'seeds':SEEDS,'scenarios':SCENARIOS,'rows':rows},indent=2)+'\\n')\n\n    for name in MODELS:\n        group=[r for r in rows if r['model']==name]\n        samples=[r for r in group if r['seed']==SEEDS[0]]\n        applied=[e for r in samples for e in r['events'][1:] if e['status']=='applied']\n        print('G1_SUMMARY:',json.dumps({\n            'model':name,'runs':len(group),\n            'applied':len(applied),'spoken_dynamic_events':len(samples)*2,\n            'mean_queue_s':float(np.mean([e['queue_s'] for e in applied])),\n            'max_queue_s':max(e['queue_s'] for e in applied),\n            'superseded_events':[(r['scenario'],e['intent']) for r in samples for e in r['events'] if e['status']=='superseded'],\n            'mean_path_error_cm':100*float(np.mean([r['mean_root_path_error_m'] for r in group])),\n            'mean_stop_pending_distance_m':float(np.mean([r['stop_ready_to_apply_path_m'] for r in group])),\n            'mean_post_stop_target_error_cm':100*float(np.mean([r['post_stop_mean_target_error_m'] for r in group])),\n            'representative_window_ms':1000*float(np.median([r['median_window_generation_s'] for r in group])),\n            'peak_allocated_mib':max(r['cuda_peak_mib'] for r in group),\n            'all_qpos_shape':[r.get('qpos_shape') for r in samples],\n            'policy':group[0]['policy'], 'mean_generated_to_played_ratio':float(np.mean([r['generated_to_played_ratio'] for r in group])),\n            'mean_stop_end_to_new_motion_s':float(np.mean([r['events'][-1]['estimated_aud_end_to_motion_ready_s'] for r in group])),\n            'max_stop_queue_s':max(r['events'][-1]['queue_s'] for r in group),\n        }),flush=True)\n\n\nif __name__ == '__main__':\n    main()\n", encoding='utf-8')
import subprocess
from pathlib import Path
root = Path('/content/ardy-repro')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/"replan_g1.py"), *[]], cwd=root, check=True)


## 社区接口和动力仿真

先逐帧加载 G1 参考，通过社区适配器生成 50 Hz 的 SONIC 参考。下一个单元运行 NVIDIA ONNX 策略和 HumanTracker 的 MuJoCo 后端。


In [ ]:
from pathlib import Path
root = Path('/content/ardy-repro')
root.mkdir(exist_ok=True)
(root / "check_community.py").write_text("\"\"\"Validate ARDY's G1 qpos against the official model and the community SONIC adapter.\"\"\"\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport urllib.request\n\nos.environ.setdefault('MUJOCO_GL', 'egl')\nimport mujoco\nimport numpy as np\nfrom ardy.assets import skeleton_asset_path\n\nWORK = Path('/content/ardy-repro')\nRESULTS = WORK / 'results'\nCOMMUNITY_REV = 'b75d8c56b65e05857146f7e65dd4f87caeb78ed8'\nBRIDGE = WORK / 'sonic-g1-bridge'\n\n\ndef prepare_bridge():\n    if (BRIDGE / 'scripts/run_g1_sonic.py').is_file():\n        return\n    archive = WORK / 'community-bridge.tar.gz'\n    url = f'https://codeload.github.com/liylo/ardy/tar.gz/{COMMUNITY_REV}'\n    urllib.request.urlretrieve(url, archive)\n    BRIDGE.mkdir(exist_ok=True)\n    subprocess.run(['tar', '-xzf', str(archive), '-C', str(BRIDGE), '--strip-components=1'], check=True)\n\n\ndef verify(model, csv):\n    qs = np.loadtxt(csv, delimiter=',', ndmin=2)\n    if qs.shape != (240, 36) or model.nq != 36 or not np.isfinite(qs).all():\n        raise ValueError(f'Invalid qpos in {csv.name}: {qs.shape} for model.nq {model.nq}')\n    joints = np.flatnonzero(\n        (model.jnt_type == mujoco.mjtJoint.mjJNT_HINGE) & model.jnt_limited.astype(bool)\n    )\n    adr = model.jnt_qposadr[joints]\n    values = qs[:, adr]\n    limits = model.jnt_range[joints]\n    violations = np.maximum(np.maximum(limits[:,0] - values, values - limits[:,1]), 0)\n    peak_joint = np.max(violations, axis=0)\n    worst = int(np.argmax(peak_joint))\n    data = mujoco.MjData(model)\n    for q in qs:\n        data.qpos[:] = q\n        mujoco.mj_forward(model, data)\n        if not np.isfinite(data.xpos).all():\n            raise ValueError(f'Non-finite kinematics for {csv.name}')\n    return qs, {\n        'case':csv.stem,'qpos_shape':list(qs.shape),\n        'kinematic_frames':len(qs),'joint_limit_checked':len(joints),\n        'frames_above_limits':int((violations.max(axis=1)>1e-3).sum()),\n        'joint_frame_violations':int((violations>1e-3).sum()),\n        'peak_limit_error_rad':float(peak_joint[worst]),\n        'worst_limit_joint':mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_JOINT,int(joints[worst])),\n        'peak_joint_speed_rad_s':float(np.abs(np.diff(qs[:,7:],axis=0)).max()*25),\n        'max_root_quat_norm_error':float(np.abs(np.linalg.norm(qs[:,3:7],axis=1)-1).max()),\n    }\n\n\ndef playback_video(model, qs, output):\n    data=mujoco.MjData(model)\n    height,width=360,640\n    cam=mujoco.MjvCamera()\n    cam.lookat=qs[:,:3].mean(axis=0)\n    cam.lookat[2]=.75\n    cam.distance=4.45\n    cam.azimuth=130\n    cam.elevation=-14\n    command=['ffmpeg','-y','-v','error','-f','rawvideo','-pix_fmt','rgb24',\n             '-s',f'{width}x{height}','-r','25','-i','-','-an','-c:v','libx264',\n             '-pix_fmt','yuv420p','-crf','26',str(output)]\n    with mujoco.Renderer(model,height=height,width=width) as renderer:\n        with subprocess.Popen(command,stdin=subprocess.PIPE) as encoder:\n            for q in qs:\n                data.qpos[:]=q\n                mujoco.mj_forward(model,data)\n                renderer.update_scene(data,camera=cam)\n                encoder.stdin.write(renderer.render().tobytes())\n            encoder.stdin.close()\n            if encoder.wait():\n                raise RuntimeError('Failed to encode the kinematic playback')\n\n\ndef main():\n    xml=skeleton_asset_path('g1skel34','xml','g1.xml')\n    model=mujoco.MjModel.from_xml_path(str(xml))\n    prepare_bridge()\n    source=RESULTS/'G1_H52_after_boundary.csv'\n    coverage=[]\n    for csv in sorted(RESULTS.glob('G1_H*.csv')):\n        qs, row=verify(model,csv)\n        coverage.append(row)\n        if csv==source:\n            prepared=RESULTS/'G1_H52_50hz'\n            subprocess.run([\n                sys.executable,str(BRIDGE/'scripts/run_g1_sonic.py'),\n                '--motion-csv',str(csv),'--output',str(prepared),\n            ],cwd=BRIDGE,check=True)\n            files=list(RESULTS.glob('G1_H52_50hz*sonic_g1.npz'))\n            assert len(files)==1, files\n            with np.load(files[0],allow_pickle=False) as data:\n                row['sonic_file']=files[0].name\n                row['sonic_fields']={key:list(data[key].shape) for key in data.files}\n                scalar_keys=[key for key in data.files if 'fps' in key.lower() or 'hz' in key.lower()]\n                row['sonic_rates']={key:float(data[key].reshape(-1)[0]) for key in scalar_keys if data[key].size==1}\n                assert all(np.isfinite(data[key]).all() for key in data.files if data[key].dtype.kind in 'fc')\n            try:\n                mov=RESULTS/'G1_MuJoCo_kinematic_playback.mp4'\n                playback_video(model,qs,mov)\n                row['video']=mov.name\n            except (ValueError,RuntimeError,ImportError,OSError) as error:\n                row['video_error']=type(error).__name__+': '+str(error)\n                print('Gl render unavailable:',row['video_error'],flush=True)\n        print('G1_MUJOCO:',json.dumps(row),flush=True)\n    report={\n        'mujoco_version':mujoco.__version__, 'g1_xml':'g1skel34/xml/g1.xml',\n        'sim_mode':'qpos assignment plus mj_forward',\n        'community_repository':'https://github.com/liylo/ardy',\n        'community_revision':COMMUNITY_REV,\n        'entries':coverage,\n    }\n    (RESULTS/'community_g1_validation.json').write_text(json.dumps(report,indent=2)+'\\n')\n\n\nif __name__=='__main__':\n    main()\n", encoding='utf-8')
import subprocess
from pathlib import Path
root = Path('/content/ardy-repro')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/"check_community.py"), *[]], cwd=root, check=True)


In [ ]:
from pathlib import Path
root = Path('/content/ardy-repro')
root.mkdir(exist_ok=True)
(root / "prepare_sonic.py").write_text("import concurrent.futures\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport urllib.request\n\nroot=Path('/content/ardy-repro')\ncommit='492f128dc903d1c859d2ff535c549c149f9adda8'\nsrc=root/'humantracker'\ndef source():\n    if (src/'src/humantracker/eval/backends/sonic.py').exists():\n        return\n    tar=root/'humantracker.tar.gz'\n    urllib.request.urlretrieve(f'https://codeload.github.com/GalaxyGeneralRobotics/HumanTracker/tar.gz/{commit}',tar)\n    src.mkdir(exist_ok=True)\n    subprocess.run(['tar','-xzf',str(tar),'-C',str(src),'--strip-components=1'],check=True)\n    print('Community SONIC backend prepared',commit,flush=True)\n\ndef models():\n    from huggingface_hub import HfApi, snapshot_download\n    mid='nvidia/GEAR-SONIC'\n    revision='6733128a3d8a523b1418b06bca3cdf61c8b0987f'\n    snap=snapshot_download(mid,\n         revision=revision,allow_patterns=['model_encoder.onnx','model_decoder.onnx','LICENSE','observation_config.yaml'],\n         local_dir=root/'sonic_onnx',token=False,max_workers=2)\n    (root/'sonic_onnx_revision.txt').write_text(revision+'\\n')\n    print('Pinned encoder/decoder',revision,flush=True)\n\nwith concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:\n    tasks=[\n        pool.submit(source),pool.submit(models),\n        pool.submit(subprocess.run,['uv','pip','install','--python',sys.executable,'opencv-python-headless==4.11.0.86','imageio','numpy==1.26.4','pandas==2.2.3','tqdm'],check=True)\n    ]\n    for task in tasks:task.result()\nenv=os.environ.copy()\nenv['PYTHONPATH']=str(src/'src')\nsubprocess.run([sys.executable,'-c','from humantracker.eval.backends.sonic import SonicOnnxPolicy, SonicMjSim, SonicObsBuilder; print(\"SONIC CPU backend imports\")'],env=env,check=True)\n", encoding='utf-8')
import subprocess
from pathlib import Path
root = Path('/content/ardy-repro')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/"prepare_sonic.py"), *[]], cwd=root, check=True)
from pathlib import Path
root = Path('/content/ardy-repro')
root.mkdir(exist_ok=True)
(root / "run_sonic_sim.py").write_text("\"\"\"Track synthetic ARDY G1 references with NVIDIA SONIC in CPU MuJoCo.\"\"\"\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nos.environ.setdefault('MUJOCO_GL','egl')\nimport mujoco\nimport numpy as np\nfrom scipy.spatial.transform import Rotation, Slerp\n\nBASE=Path('/content/ardy-repro')\nSRC=BASE/'humantracker'\nsys.path.insert(0,str(SRC/'src'))\nfrom humantracker.eval.backends.sonic import (\n    SonicMjSim, SonicObsBuilder, SonicOnnxPolicy, action_to_motor_targets,\n    DEFAULT_ANGLES_ISAB, ISAB_TO_MUJ,\n)\nfrom humantracker.eval.core.geometry import get_sensor_data\nfrom humantracker.eval.core.gt_render import DualRobotRenderer\n\nXML=SRC/'storage/assets/unitree_g1_5010/scene_mjx_track_papergray.xml'\nOUT=BASE/'results'\nSUITES=('before_boundary','after_boundary','rapid_override')\n\n\ndef to_50hz(q, model):\n    source=np.arange(len(q),dtype=float)*.04\n    target=np.arange(2*len(q)-1,dtype=float)*.02\n    q50=np.stack([np.interp(target,source,q[:,j]) for j in range(36)],axis=1)\n    q50[:,3:7]=Slerp(source,Rotation.from_quat(q[:,3:7],scalar_first=True))(target).as_quat(scalar_first=True)\n    q50[:,:2]-=q50[:1,:2].copy()\n    vel=np.zeros((len(q50),model.nv))\n    for i in range(len(vel)-1):\n        mujoco.mj_differentiatePos(model,vel[i],.02,q50[i],q50[i+1])\n    vel[-1]=vel[-2]\n    return q50, vel\n\n\ndef track(qcsv, policy):\n    model=mujoco.MjModel.from_xml_path(str(XML))\n    ref,vel=to_50hz(np.loadtxt(qcsv,delimiter=','),model)\n    sim=SonicMjSim(init_qpos=ref[0].copy(),headless=True,xml_path=str(XML))\n    state=sim.reset(sim.init_state())\n    obs=SonicObsBuilder({'qpos':ref,'qvel':vel},encoder_dim=policy.encoder_input_dim)\n    obs.reset(state.mj_data.qpos[3:7].copy())\n    last=np.zeros(29,np.float32)\n    poses=[]\n    for i in range(len(ref)):\n        q=state.mj_data.qpos.copy()\n        dq=state.mj_data.qvel.copy()\n        gyro=get_sensor_data(sim.mj_model,state.mj_data,'gyro_pelvis')\n        obs.update_history(gyro,q[7:][ISAB_TO_MUJ]-DEFAULT_ANGLES_ISAB,dq[6:][ISAB_TO_MUJ],last,q[3:7])\n        token=policy.encode(obs.build_encoder_obs(i,q[3:7]))\n        last=policy.decode(obs.build_decoder_obs(token))[0].copy()\n        state=sim.step(state,action_to_motor_targets(last))\n        poses.append(state.mj_data.qpos.copy())\n    actual=np.stack(poses)\n    assert actual.shape==ref.shape and np.isfinite(actual).all()\n    root=np.linalg.norm(actual[:,:2]-ref[:,:2],axis=1)\n    ang=actual[:,7:]-ref[:,7:]\n    wrap=np.arctan2(np.sin(ang),np.cos(ang))\n    low=actual[:,2]<.35\n    fallen=np.flatnonzero(np.convolve(low,np.ones(5,dtype=int),mode='valid')==5)\n    out={\n        'case':qcsv.stem,'control_frames':len(ref),'fps':50,'mujoco_timestep_s':float(sim.mj_model.opt.timestep),\n        'root_xy_mae_m':float(root.mean()),'root_xy_final_m':float(root[-1]),\n        'joint_mae_deg':float(np.degrees(np.abs(wrap).mean())),\n        'pelvis_min_z_m':float(actual[:,2].min()),\n        'pelvis_below_035m_5frames_s':None if not len(fallen) else float(fallen[0]/50),\n    }\n    np.savez_compressed(OUT/(qcsv.stem+'_closed_loop.npz'),reference=ref,actual=actual,fps=50)\n    return out,ref,actual\n\n\ndef render(ref,actual,output):\n    width,height=640,360\n    renderer=DualRobotRenderer(str(XML),width,height)\n    cmd=['ffmpeg','-y','-v','error','-f','rawvideo','-pix_fmt','rgb24','-s',f'{width}x{height}',\n         '-r','25','-i','-','-an','-c:v','libx264','-pix_fmt','yuv420p','-crf','27',str(output)]\n    try:\n        with subprocess.Popen(cmd,stdin=subprocess.PIPE) as proc:\n            for i in range(0,len(ref),2):\n                image=renderer.render(actual[i],ref[i])\n                if image.shape!=(height,width,3): raise ValueError(f'Renderer returned {image.shape}')\n                proc.stdin.write(np.ascontiguousarray(image).tobytes())\n            proc.stdin.close()\n            if proc.wait(): raise RuntimeError('FFmpeg render failed')\n    finally:\n        renderer.close()\n\n\ndef main():\n    policy= SonicOnnxPolicy(str(BASE/'sonic_onnx/model_encoder.onnx'),\n                           str(BASE/'sonic_onnx/model_decoder.onnx'),device='cpu')\n    print('Loaded SONIC policy widths:',policy.encoder_input_dim,flush=True)\n    rows=[]\n    rendered=None\n    for prefix in ('G1_H52','G1_H52_patch_52lookahead'):\n        for case in SUITES:\n            csv=OUT/f'{prefix}_{case}.csv'\n            if not csv.exists():\n                continue\n            result,ref,actual=track(csv,policy)\n            if prefix=='G1_H52' and case=='after_boundary':\n                rendered=ref,actual\n            rows.append(result)\n            print('SONIC_CLOSED:',json.dumps(result),flush=True)\n            (OUT/'sonic_closed_loop.json').write_text(json.dumps({\n                'humantracker_commit':'492f128dc903d1c859d2ff535c549c149f9adda8',\n                'sonic_revision':(BASE/'sonic_onnx_revision.txt').read_text().strip(),\n                'provider':'CPUExecutionProvider','cases':rows,\n            },indent=2)+'\\n')\n    if rendered is not None and os.environ.get('ARDY_RENDER_DUAL') == '1':\n        try:\n            render(*rendered,OUT/'G1_SONIC_closed_loop.mp4')\n            print('Closed loop video exported',flush=True)\n        except (ValueError,RuntimeError,ImportError,OSError) as error:\n            print('Closed loop render error:',type(error).__name__,str(error),flush=True)\n\n\nif __name__=='__main__':\n    main()\n", encoding='utf-8')
import subprocess
from pathlib import Path
root = Path('/content/ardy-repro')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/"run_sonic_sim.py"), *[]], cwd=root, check=True)

(root / "plot_sonic_trace.py").write_text("import numpy as np\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nfrom matplotlib.animation import FFMpegWriter\nfrom pathlib import Path\nimport subprocess\nroot=Path('/content/ardy-repro/results')\ndata=np.load(root/'G1_H52_after_boundary_closed_loop.npz')\nref,actual=data['reference'],data['actual']\nassert ref.shape==actual.shape and np.isfinite(actual).all()\ntimes=np.arange(len(ref))/50\nerror=np.linalg.norm(ref[:,:2]-actual[:,:2],axis=1)\nfig,(xy,err)=plt.subplots(1,2,figsize=(8,3.75))\nfig.subplots_adjust(left=.1,right=.98,bottom=.17,top=.84,wspace=.33)\nxy.plot(ref[:,0],ref[:,1],':',color='#9da8ac',label='Reference route',lw=1)\ntraj_ref,=xy.plot([],[],color='#236aa5',label='Reference',lw=2)\ntraj_act,=xy.plot([],[],color='#bb622b',label='SONIC / MuJoCo',lw=2)\nxy.set(xlabel='X (m)',ylabel='Y (m)',title='Ground-plane position')\npts=np.r_[ref[:,:2],actual[:,:2]]\nxy.set(xlim=(pts[:,0].min()-.25,pts[:,0].max()+.25),ylim=(pts[:,1].min()-.25,pts[:,1].max()+.25))\nxy.set_aspect('equal')\nxy.legend(loc='best',fontsize=7)\nerr.plot(times,error,color='#dce2e3')\ntrace,=err.plot([],[],color='#bb622b',lw=2)\nerr.set(xlabel='Replayed time (s)',ylabel='Root error (m)',title='Tracking error',xlim=(0,times[-1]),ylim=(0,max(error)*1.05))\nerr.grid(alpha=.25)\ntitle=fig.suptitle('')\ndef update(i):\n    traj_ref.set_data(ref[:i+1,0],ref[:i+1,1])\n    traj_act.set_data(actual[:i+1,0],actual[:i+1,1])\n    trace.set_data(times[:i+1],error[:i+1])\n    title.set_text(f'SONIC closed-loop in MuJoCo   |   {times[i]:.2f} s')\nout=root/'G1_SONIC_tracking_plot.mp4'\nwriter=FFMpegWriter(fps=25,codec='libx264',extra_args=['-pix_fmt','yuv420p','-preset','veryfast','-crf','30'])\nwith writer.saving(fig,str(out),96):\n    for i in range(0,len(times),2):\n        update(i)\n        writer.grab_frame()\nplt.close(fig)\nsubprocess.run(['ffprobe','-v','error','-show_entries','stream=codec_type,codec_name,nb_frames:format=duration,size','-of','json',str(out)],check=True)\n", encoding='utf-8')
subprocess.run([str(root/'python/bin/python'), '-u', str(root/'plot_sonic_trace.py')], cwd=root, check=True)


## 保存产物

挂载 Drive 后把 `results/` 复制进任务目录。Colab 临时盘存放权重和环境。


In [ ]:
from pathlib import Path
import shutil
from google.colab import drive
drive.mount('/content/drive')
target = Path('/content/drive/MyDrive/语音控制动作生成/ardy/Colab复跑结果')
shutil.copytree('/content/ardy-repro/results', target, dirs_exist_ok=True)
print(target)
